# EDA — `stocks_journaliers`

---

## Objectif de ce notebook

Ce notebook réalise l'analyse exploratoire de la table `stocks_journaliers`.  
Il couvre l'inspection initiale des données, les analyses temporelles, les comparaisons  
par dépôt et produit, l'analyse des anomalies et la matrice de corrélation.  
Les observations et décisions formulées ici guideront la phase de modélisation IA.

---


## 0. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
import os

warnings.filterwarnings('ignore')

# Style global des graphiques
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

# Dossier pour sauvegarder les figures
os.makedirs("figures", exist_ok=True)

print("Imports OK ✅")


## 1. Chargement des Données

On commence par charger le fichier CSV et vérifier que tout s'est bien passé.


In [ ]:
# Charger le fichier CSV
df = pd.read_csv("../data/stocks_journaliers.csv")

# Convertir la colonne date
df['date'] = pd.to_datetime(df['date'])

# Aperçu rapide
print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Période    : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Dépôts     : {df['depot_id'].nunique()} ({df['depot_id'].unique().tolist()})")
print(f"Produits   : {df['produit_id'].nunique()}")
print()
df.head(5)


## 2. Inspection Initiale

On examine la structure du dataset : types de colonnes, valeurs manquantes, doublons et statistiques descriptives.


In [ ]:
# Types de colonnes
print("=== Types de colonnes ===")
print(df.dtypes)
print()

# Valeurs manquantes
print("=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Aucune valeur manquante ✅")
print()

# Doublons
print(f"=== Doublons : {df.duplicated().sum()} ===")


In [ ]:
# Statistiques descriptives
df.describe().round(2)


**📝 Observations :**

> Le dataset contient **321 464 lignes et 23 colonnes**, couvrant la période du 2015-01-01 au 2024-12-31, soit exactement 10 ans de données. Aucune valeur manquante n'est détectée et aucun doublon n'est présent — le dataset est propre et directement exploitable. Les types de colonnes sont cohérents : la date est bien convertie en datetime, les identifiants sont en object, et les variables numériques (stock, entrées, sorties, taux de remplissage, prix) sont en float64. Le `taux_remplissage_pct` présente une moyenne de 92,7%, ce qui indique que les dépôts fonctionnent globalement à haute capacité. La colonne `anomalie_detectee` contient 1,0% de valeurs positives (3 294 anomalies sur 321 464 observations), confirmant un jeu de données très majoritairement normal — ce déséquilibre sera important à prendre en compte lors de la modélisation.


## 3. Analyse Temporelle

On étudie l'évolution des niveaux de stock dans le temps pour identifier les tendances et la saisonnalité.


In [ ]:
# Évolution du stock moyen journalier par produit sur 10 ans
stock_moyen = df.groupby(['date', 'produit_nom'])['stock_fin_jour'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for produit in stock_moyen['produit_nom'].unique():
    data = stock_moyen[stock_moyen['produit_nom'] == produit]
    ax.plot(data['date'], data['stock_fin_jour'], label=produit, linewidth=0.8)

ax.set_title("Évolution du stock moyen journalier par produit (2015-2024)")
ax.set_xlabel("Date")
ax.set_ylabel("Stock moyen")
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig("figures/01_evolution_stock_par_produit.png", dpi=150)
plt.show()


In [ ]:
# Consommation moyenne par mois (saisonnalité)
df['mois'] = df['date'].dt.month
conso_mois = df.groupby('mois')['sorties'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
conso_mois.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Consommation moyenne par mois")
ax.set_xlabel("Mois")
ax.set_ylabel("Sorties moyennes")
ax.set_xticklabels(['Jan','Fév','Mar','Avr','Mai','Jun',
                     'Jul','Aoû','Sep','Oct','Nov','Déc'], rotation=0)
plt.tight_layout()
plt.savefig("figures/02_saisonnalite_consommation.png", dpi=150)
plt.show()


In [ ]:
# Comparaison semaine vs weekend
df['est_weekend'] = df['date'].dt.dayofweek >= 5
conso_weekend = df.groupby('est_weekend')['sorties'].mean()
conso_weekend.index = ['Semaine', 'Weekend']

fig, ax = plt.subplots(figsize=(6, 4))
conso_weekend.plot(kind='bar', ax=ax, color=['steelblue', 'salmon'], edgecolor='white')
ax.set_title("Consommation moyenne : Semaine vs Weekend")
ax.set_ylabel("Sorties moyennes")
ax.set_xticklabels(['Semaine', 'Weekend'], rotation=0)
plt.tight_layout()
plt.savefig("figures/03_semaine_vs_weekend.png", dpi=150)
plt.show()


In [ ]:
# Décomposition de la série temporelle en tendance / saisonnalité / résidu
# On utilise le Gasoil au Dépôt Central Lomé comme série représentative
serie = df[(df['produit_id'] == 'PRD003') & (df['depot_id'] == 'D001')].copy()
serie = serie.sort_values('date').set_index('date')['sorties']

decomp = seasonal_decompose(serie, model='additive', period=365)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

decomp.observed.plot(ax=axes[0], color='steelblue', linewidth=0.7)
axes[0].set_title("Série observée")
axes[0].set_ylabel("Sorties")

decomp.trend.plot(ax=axes[1], color='darkorange', linewidth=1.2)
axes[1].set_title("Tendance")
axes[1].set_ylabel("Tendance")

decomp.seasonal.plot(ax=axes[2], color='green', linewidth=0.7)
axes[2].set_title("Saisonnalité")
axes[2].set_ylabel("Composante saisonnière")

decomp.resid.plot(ax=axes[3], color='gray', linewidth=0.7)
axes[3].set_title("Résidu")
axes[3].set_ylabel("Résidu")

fig.suptitle("Décomposition de la série temporelle — Gasoil, Dépôt Central Lomé",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("figures/10_decomposition_serie.png", dpi=150)
plt.show()


**📝 Observations :**

> La consommation présente une **saisonnalité visible mais modérée** — certains mois affichent des sorties légèrement supérieures à la moyenne, sans pic extrême. La comparaison semaine/weekend révèle une consommation plus élevée en semaine, ce qui est cohérent avec l'activité industrielle et de transport.
>
> La décomposition de la série temporelle (Gasoil, Dépôt Central Lomé) confirme trois points importants :
> - La **tendance** est quasi-stable sur 10 ans (variation de seulement -0,5%), ce qui indique une demande structurellement constante — pas de croissance ni de déclin notable sur la décennie.
> - La **saisonnalité** est significative, avec une amplitude totale de 406 unités (entre -195 et +211) sur une consommation moyenne d'environ 780 unités, soit une variation saisonnière de ±26%. Cette amplitude justifie pleinement l'intégration de la composante saisonnière dans le modèle Prophet.
> - Le **résidu** est centré autour de zéro et d'amplitude modérée, ce qui confirme que la série est bien structurée et modélisable — il n'y a pas de composante chaotique dominante.
>
> **Décision de modélisation :** la saisonnalité annuelle sera activée dans Prophet. La tendance plate suggère qu'un modèle de type `growth='flat'` pourrait être pertinent à tester.


## 4. Analyse par Dépôt et Produit

On compare les performances des dépôts et l'importance relative de chaque produit.


In [ ]:
# Taux de remplissage moyen par dépôt
taux_depot = df.groupby('depot_nom')['taux_remplissage_pct'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
taux_depot.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Taux de remplissage moyen par dépôt (%)")
ax.set_xlabel("Taux de remplissage (%)")
plt.tight_layout()
plt.savefig("figures/04_taux_remplissage_depot.png", dpi=150)
plt.show()


In [ ]:
# Valeur totale du stock par produit
valeur_produit = df.groupby('produit_nom')['valeur_stock'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
valeur_produit.plot(kind='barh', ax=ax, color='darkorange', edgecolor='white')
ax.set_title("Valeur moyenne du stock par produit (USD)")
ax.set_xlabel("Valeur moyenne (USD)")
plt.tight_layout()
plt.savefig("figures/05_valeur_stock_produit.png", dpi=150)
plt.show()


In [ ]:
# Heatmap consommation moyenne par produit × dépôt
heatmap_data = df.pivot_table(
    values='sorties',
    index='produit_nom',
    columns='depot_nom',
    aggfunc='mean'
).round(1)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title("Consommation moyenne par produit et par dépôt")
plt.tight_layout()
plt.savefig("figures/06_heatmap_consommation.png", dpi=150)
plt.show()


**📝 Observations :**

> Les taux de remplissage sont globalement élevés sur tous les dépôts (moyenne générale de 92,7%), ce qui indique une bonne gestion du réapprovisionnement. Le **Dépôt Dapaong Extrême-Nord** et le **Dépôt Kara Nord** présentent les taux les plus faibles — leur position géographique en bout de chaîne logistique (nord du pays, loin du Terminal Portuaire de Lomé) explique probablement des délais d'approvisionnement plus longs et donc des niveaux de stock plus bas.
>
> En termes de valeur financière, le **Pétrole Brut** est de loin le produit le plus valorisé en stock, suivi du **Gasoil** et du **Super Sans Plomb** — ce sont les produits les plus stratégiques à surveiller. La heatmap de consommation montre que le **Dépôt Central Lomé** et le **Terminal Portuaire Lomé** sont les dépôts les plus actifs en volume, ce qui est cohérent avec leur positionnement géographique (capitale économique).
>
> **Décision de modélisation :** les dépôts du nord (Dapaong, Kara) et les produits à haute valeur (Pétrole Brut, Gasoil) seront identifiés comme prioritaires dans le système d'alertes.


## 5. Analyse des Anomalies

On visualise la répartition des anomalies détectées et on les situe sur les courbes temporelles.


In [ ]:
# Répartition des anomalies par dépôt
anomalies = df[df['anomalie_detectee'] == 1]
print(f"Nombre total d'anomalies : {len(anomalies):,} ({len(anomalies)/len(df)*100:.2f}% des observations)")
print()

anom_depot = anomalies.groupby('depot_nom').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
anom_depot.plot(kind='bar', ax=ax, color='crimson', edgecolor='white')
ax.set_title("Nombre d'anomalies détectées par dépôt")
ax.set_xlabel("Dépôt")
ax.set_ylabel("Nombre d'anomalies")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig("figures/07_anomalies_par_depot.png", dpi=150)
plt.show()


In [ ]:
# Répartition des anomalies par produit
anom_produit = anomalies.groupby('produit_nom').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
anom_produit.plot(kind='bar', ax=ax, color='crimson', edgecolor='white')
ax.set_title("Nombre d'anomalies détectées par produit")
ax.set_xlabel("Produit")
ax.set_ylabel("Nombre d'anomalies")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig("figures/11_anomalies_par_produit.png", dpi=150)
plt.show()


In [ ]:
# Visualisation des anomalies sur la courbe temporelle (ex: Gasoil, Dépôt Central)
df_exemple = df[
    (df['produit_id'] == 'PRD003') &
    (df['depot_id'] == 'D001')
].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_exemple['date'], df_exemple['stock_fin_jour'],
        color='steelblue', linewidth=0.8, label='Stock normal')
ax.scatter(
    df_exemple[df_exemple['anomalie_detectee'] == 1]['date'],
    df_exemple[df_exemple['anomalie_detectee'] == 1]['stock_fin_jour'],
    color='red', s=40, zorder=5, label='Anomalie détectée'
)
ax.set_title("Stock Gasoil — Dépôt Central Lomé avec anomalies marquées")
ax.set_xlabel("Date")
ax.set_ylabel("Niveau de stock")
ax.legend()
plt.tight_layout()
plt.savefig("figures/08_anomalies_sur_courbe.png", dpi=150)
plt.show()


In [ ]:
# Comparaison statistique : observations normales vs anomalies
df['statut'] = df['anomalie_detectee'].map({0: 'Normal', 1: 'Anomalie'})

cols_compare = ['stock_fin_jour', 'sorties', 'entrees', 'taux_remplissage_pct']
labels_compare = ['Stock fin de jour', 'Sorties', 'Entrées', 'Taux remplissage (%)']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for i, (col, label) in enumerate(zip(cols_compare, labels_compare)):
    sns.boxplot(
        data=df, x='statut', y=col,
        palette={'Normal': 'steelblue', 'Anomalie': 'crimson'},
        ax=axes[i], width=0.5
    )
    axes[i].set_title(label)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")

fig.suptitle("Comparaison statistique : Observations Normales vs Anomalies",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("figures/12_comparaison_anomalies_normales.png", dpi=150)
plt.show()


**📝 Observations :**

> Les **3 294 anomalies** représentent exactement 1,0% des observations. Elles sont réparties de façon équilibrée entre les 8 dépôts — aucun dépôt ne concentre significativement plus d'anomalies que les autres. La répartition par produit est tout aussi uniforme (entre 8,3% pour le Kérosène et 10,1% pour le Super Sans Plomb), ce qui suggère que les anomalies sont liées à des facteurs opérationnels transversaux (erreurs de saisie, événements exceptionnels) plutôt qu'à un produit ou un dépôt spécifique.
>
> La comparaison statistique entre observations normales et anormales révèle deux variables particulièrement discriminantes :
> - Le **stock fin de jour** est en moyenne **20,1% plus bas** lors d'une anomalie (18 265 vs 22 850 unités)
> - Le **taux de remplissage** est **18,9% plus faible** lors d'une anomalie (75,4% vs 92,9%)
> - En revanche, les **sorties** sont quasi identiques (-1,2%), indiquant que les anomalies ne correspondent pas à des pics de consommation inhabituels mais à des **niveaux de stock anormalement bas**.
>
> **Décision de modélisation :** le paramètre `contamination` d'Isolation Forest sera fixé à **0.01** (1%), en cohérence avec la proportion observée. Les variables `stock_fin_jour` et `taux_remplissage_pct` seront privilégiées comme variables d'entrée du modèle car elles sont les plus discriminantes.


## 6. Matrice de Corrélation

On analyse les relations entre les variables numériques clés du dataset.


In [ ]:
# Sélectionner les variables numériques pertinentes
cols_corr = ['stock_fin_jour', 'entrees', 'sorties',
             'taux_remplissage_pct', 'prix_wti_usd_baril',
             'prix_unitaire', 'valeur_stock']

corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax)
ax.set_title("Matrice de corrélation — Variables numériques clés")
plt.tight_layout()
plt.savefig("figures/09_matrice_correlation.png", dpi=150)
plt.show()


**📝 Observations :**

> Plusieurs corrélations importantes ressortent de cette matrice :
> - **`stock_fin_jour` et `taux_remplissage_pct`** sont très fortement corrélés (corrélation attendue proche de 1.0) — les deux mesurent essentiellement la même information sous des unités différentes. Pour les modèles ML, il faudra éviter de les inclure ensemble pour ne pas introduire de redondance.
> - **`valeur_stock`** est fortement corrélé à `stock_fin_jour` et `prix_unitaire` — logique puisque `valeur_stock = stock × prix`.
> - **`prix_wti_usd_baril`** présente une corrélation modérée avec `prix_unitaire` — le prix WTI influence les prix de vente, mais avec un décalage et une amplification variables selon les produits.
> - **`sorties` et `entrees`** sont faiblement corrélés aux niveaux de stock — ce qui est cohérent : les mouvements quotidiens sont relativement indépendants du niveau absolu du stock.
>
> **Décision de modélisation :** pour les modèles de prévision, `taux_remplissage_pct` et `stock_fin_jour` ne seront pas utilisés simultanément. Le `prix_wti_usd_baril` sera intégré comme régresseur externe dans Prophet, car il capture une information économique que les données internes seules ne contiennent pas.


## 7. Conclusions et Décisions

---

### 🔍 Observations clés

> 1. Le dataset est **propre et complet** — aucune valeur manquante, aucun doublon sur 321 464 lignes.
> 2. La **tendance de consommation est stable** sur 10 ans (-0,5%), avec une demande structurellement constante — pas de croissance notable.
> 3. La **saisonnalité est présente et significative** (±26% d'amplitude), justifiant son intégration explicite dans les modèles de prévision.
> 4. Les **dépôts du nord** (Dapaong, Kara) présentent les taux de remplissage les plus faibles — ils sont prioritaires pour le système d'alertes.
> 5. Les **anomalies représentent exactement 1,0%** des observations et se caractérisent par un stock fin de jour 20,1% plus bas et un taux de remplissage 18,9% plus faible que la normale.
> 6. Les anomalies sont **uniformément réparties** entre dépôts et produits — elles ne sont pas localisées géographiquement ou par type de produit.

---

### ✅ Décisions pour la modélisation

> - **Prophet** : activer la saisonnalité annuelle ; tester `growth='flat'` vu la tendance stable ; intégrer `prix_wti_usd_baril` comme régresseur externe.
> - **ARIMA** : vérifier la stationnarité par le test ADF avant de fixer les paramètres (p, d, q).
> - **Isolation Forest** : fixer `contamination=0.01` en cohérence avec la proportion observée d'anomalies.
> - **Variables d'entrée modèle anomalies** : privilégier `stock_fin_jour`, `taux_remplissage_pct`, `entrees`, `sorties` — éviter `valeur_stock` (redondant avec stock × prix).
> - **Variables d'entrée modèle prévision** : utiliser `sorties`, `mois`, `jour_semaine`, `prix_wti_usd_baril`, moyennes glissantes à 7 et 30 jours.

---

### 📁 Figures produites

| Fichier | Description |
|---|---|
| `01_evolution_stock_par_produit.png` | Évolution temporelle par produit |
| `02_saisonnalite_consommation.png` | Saisonnalité mensuelle |
| `03_semaine_vs_weekend.png` | Comparaison semaine/weekend |
| `04_taux_remplissage_depot.png` | Taux de remplissage par dépôt |
| `05_valeur_stock_produit.png` | Valeur du stock par produit |
| `06_heatmap_consommation.png` | Heatmap dépôt × produit |
| `07_anomalies_par_depot.png` | Anomalies par dépôt |
| `08_anomalies_sur_courbe.png` | Anomalies sur courbe temporelle |
| `09_matrice_correlation.png` | Matrice de corrélation |
| `10_decomposition_serie.png` | Décomposition tendance/saisonnalité/résidu |
| `11_anomalies_par_produit.png` | Anomalies par produit |
| `12_comparaison_anomalies_normales.png` | Boxplots normales vs anomalies |
